In [ ]:
# In[1]:

import pandas as pd
import pytz
import numpy as np

# 1. Load metric.csv
df = pd.read_csv('metric.csv')

# Convert timestamp (seconds) to timezone-aware datetimes in UTC then to Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
df['dt_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df['dt'] = df['dt_utc'].dt.tz_convert(tz)

# 1) Report counts
n_cmdb = df['cmdb_id'].nunique()
n_kpi = df['kpi_name'].nunique()
summary_counts = pd.DataFrame({
    'unique_cmdb_count': [n_cmdb],
    'unique_kpi_count': [n_kpi]
})

# 2) Compute global P95 per (cmdb_id, kpi_name) using full series (do NOT filter first)
global_p95 = df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value': 'global_P95'})

# merge global P95 back into the full df (thresholds computed globally)
df = df.merge(global_p95, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Filter metric rows to the incident window: 2022-03-21 08:30:00 to 09:00:00 UTC+8
start = pd.Timestamp('2022-03-21 08:30:00', tz=tz)
end = pd.Timestamp('2022-03-21 09:00:00', tz=tz)
df_window = df[(df['dt'] >= start) & (df['dt'] <= end)].copy()

# 4) For each pair compute anomalies count (value >= global_P95), earliest timestamp of such a point, and max value in window
if not df_window.empty:
    def agg_fn(g):
        mask = g['value'] >= g['global_P95']
        return pd.Series({
            'anomalies_in_window_count': int(mask.sum()),
            'earliest_anomaly_timestamp': (g.loc[mask, 'dt'].min() if mask.any() else pd.NaT),
            'max_value_in_window': float(g['value'].max())
        })
    agg = df_window.groupby(['cmdb_id', 'kpi_name']).apply(agg_fn).reset_index()
else:
    # empty window -> empty aggregation
    agg = pd.DataFrame(columns=['cmdb_id','kpi_name','anomalies_in_window_count','earliest_anomaly_timestamp','max_value_in_window'])

# attach global_P95
result = agg.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# format earliest_anomaly_timestamp as string for compact display
if 'earliest_anomaly_timestamp' in result.columns and not result.empty:
    result['earliest_anomaly_timestamp'] = result['earliest_anomaly_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 5) Return top 20 sorted by anomalies_in_window_count desc
if (result['anomalies_in_window_count'].sum() if not result.empty else 0) > 0:
    top_table = result.sort_values(by=['anomalies_in_window_count', 'max_value_in_window'], ascending=[False, False]) \
                      [['cmdb_id','kpi_name','global_P95','anomalies_in_window_count','earliest_anomaly_timestamp','max_value_in_window']].head(20)
else:
    # 6) If no anomalies, return top 20 pairs by max_value_in_window descending instead.
    # Note: consider only pairs that have at least one point in the window (agg may be empty)
    top_table = result.sort_values(by='max_value_in_window', ascending=False) \
                      [['cmdb_id','kpi_name','global_P95','anomalies_in_window_count','earliest_anomaly_timestamp','max_value_in_window']].head(20)

# Keep numeric columns compact
if not top_table.empty:
    top_table['global_P95'] = top_table['global_P95'].round(6)
    top_table['max_value_in_window'] = top_table['max_value_in_window'].round(6)

# Final compact outputs: counts summary and the top_table (<=20 rows)
summary_counts, top_table

```
Out[1]:
```
```python
summary = (
    "Unique cmdb_id: 59; unique kpi_name: 1224.\n"
    "Incident window: 2022-03-21 08:30:00 to 09:00:00 (UTC+8).\n"
    "Using global P95 thresholds, anomalies were detected. The returned top 20 (cmdb_id, kpi_name) all have anomalies_in_window_count = 30\n"
    "(i.e., every sampled point in the window >= global P95). The earliest anomaly for these is 2022-03-21 08:30:00+0800, and for these\n"
    "pairs the max value in the window equals the reported global_P95.\n"
    "Notable examples from the top 20: adservice2 (runtime.java_lang_Runtime_StartTime), adservice (runtime.java_lang_Runtime_StartTime),\n"
    "node-1/2/3/4/6 (system.disk.total), and many pods reporting container.node-6.container_start_time_seconds across multiple services\n"
    "(e.g., adservice2-0, cartservice2-0, cartservice-0/1/2, productcatalogservice2-0, currencyservice2-0, paymentservice2-0,\n"
    "recommendationservice2-0, frontend2-0, productcatalogservice-1/2).\n"
    "Summary conclusion: multiple node- and container-level KPIs met or exceeded their global P95 for the entire incident window; the\n"
    "first exceedance was at the window start (08:30:00+0800)."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(   unique_cmdb_count  unique_kpi_count
0                 59              1224,                        cmdb_id                                           kpi_name    global_P95  anomalies_in_window_count earliest_anomaly_timestamp  max_value_in_window
841                 adservice2                runtime.java_lang_Runtime_StartTime  1.647822e+12                         30   2022-03-21 08:30:00+0800         1.647822e+12
250                  adservice                runtime.java_lang_Runtime_StartTime  1.647523e+12                         30   2022-03-21 08:30:00+0800         1.647523e+12
3367                    node-4                                  system.disk.total  7.616300e+09                         30   2022-03-21 08:30:00+0800         7.616300e+09
3191                    node-1                                  system.disk.total  5.867011e+09                         30   2022-03-21 08:30:00+0800         5.867011e+09
3250                    node-2                                  system.disk.total  5.867011e+09                         30   2022-03-21 08:30:00+0800         5.867011e+09
3308                    node-3                                  system.disk.total  5.867011e+09                         30   2022-03-21 08:30:00+0800         5.867011e+09
3483                    node-6                                  system.disk.total  3.617924e+09                         30   2022-03-21 08:30:00+0800         3.617924e+09
238                  adservice  runtime.java_lang_OperatingSystem_CommittedVir...  3.589956e+09                         30   2022-03-21 08:30:00+0800         3.589956e+09
987               adservice2-0      container.node-6.container_start_time_seconds  1.647822e+09                         30   2022-03-21 08:30:00+0800         1.647822e+09
1377            cartservice2-0      container.node-6.container_start_time_seconds  1.647810e+09                         30   2022-03-21 08:30:00+0800         1.647810e+09
1093             cartservice-0      container.node-5.container_start_time_seconds  1.647778e+09                         30   2022-03-21 08:30:00+0800         1.647778e+09
1189             cartservice-1      container.node-5.container_start_time_seconds  1.647778e+09                         30   2022-03-21 08:30:00+0800         1.647778e+09
1289             cartservice-2      container.node-5.container_start_time_seconds  1.647778e+09                         30   2022-03-21 08:30:00+0800         1.647778e+09
4221  productcatalogservice2-0      container.node-6.container_start_time_seconds  1.647523e+09                         30   2022-03-21 08:30:00+0800         1.647523e+09
2197        currencyservice2-0      container.node-6.container_start_time_seconds  1.647523e+09                         30   2022-03-21 08:30:00+0800         1.647523e+09
3799         paymentservice2-0      container.node-6.container_start_time_seconds  1.647523e+09                         30   2022-03-21 08:30:00+0800         1.647523e+09
4565  recommendationservice2-0      container.node-6.container_start_time_seconds  1.647523e+09                         30   2022-03-21 08:30:00+0800         1.647523e+09
3063               frontend2-0      container.node-6.container_start_time_seconds  1.647523e+09                         30   2022-03-21 08:30:00+0800         1.647523e+09
4105   productcatalogservice-2      container.node-5.container_start_time_seconds  1.647523e+09                         30   2022-03-21 08:30:00+0800         1.647523e+09
3989   productcatalogservice-1      container.node-5.container_start_time_seconds  1.647523e+09                         30   2022-03-21 08:30:00+0800         1.647523e+09)```
```

In [ ]:
# In[2]:

import pandas as pd
import pytz
import numpy as np

# Load data (reuse if already in memory is okay, but reload to ensure consistency)
df = pd.read_csv('metric.csv')

# Timezone setup (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
df['dt_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
df['dt'] = df['dt_utc'].dt.tz_convert(tz)

# 1) Compute global P95 and P5 per (cmdb_id, kpi_name) using full series
global_p95 = df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.95).rename('global_P95').reset_index()
global_p5  = df.groupby(['cmdb_id', 'kpi_name'])['value'].quantile(0.05).rename('global_P5').reset_index()
global_q = global_p95.merge(global_p5, on=['cmdb_id', 'kpi_name'], how='inner')

# 2) Filter to incident window (inclusive) in UTC+8
start = pd.Timestamp('2022-03-21 08:30:00', tz=tz)
end   = pd.Timestamp('2022-03-21 09:00:00', tz=tz)
df_window = df[(df['dt'] >= start) & (df['dt'] <= end)].copy()

# Attach global thresholds to window rows for per-group comparisons
df_window = df_window.merge(global_q, on=['cmdb_id', 'kpi_name'], how='left')

# 3) Aggregate required metrics per (cmdb_id, kpi_name) inside the window
def agg_window(g):
    cnt = int(len(g))
    gp95 = float(g['global_P95'].iloc[0])
    gp5  = float(g['global_P5'].iloc[0])
    # >= P95 side
    mask_ge = g['value'] >= gp95
    cnt_ge = int(mask_ge.sum())
    earliest_ge = g.loc[mask_ge, 'dt'].min() if cnt_ge > 0 else pd.NaT
    max_val = float(g['value'].max()) if cnt>0 else np.nan
    ratio_max_over_p95 = (max_val / gp95) if (gp95 > 0) else np.nan
    # <= P5 side
    mask_le = g['value'] <= gp5
    cnt_le = int(mask_le.sum())
    earliest_le = g.loc[mask_le, 'dt'].min() if cnt_le > 0 else pd.NaT
    min_val = float(g['value'].min()) if cnt>0 else np.nan
    ratio_min_over_p5 = (min_val / gp5) if (gp5 > 0) else np.nan
    return pd.Series({
        'count_points_in_window': cnt,
        'count_points_ge_P95': cnt_ge,
        'earliest_timestamp_ge_P95': earliest_ge,
        'max_value_in_window': max_val,
        'ratio_max_over_P95': ratio_max_over_p95,
        'count_points_le_P5': cnt_le,
        'earliest_timestamp_le_P5': earliest_le,
        'min_value_in_window': min_val,
        'ratio_min_over_P5': ratio_min_over_p5
    })

if not df_window.empty:
    agg = df_window.groupby(['cmdb_id', 'kpi_name']).apply(agg_window).reset_index()
else:
    agg = pd.DataFrame(columns=[
        'cmdb_id','kpi_name','count_points_in_window','count_points_ge_P95','earliest_timestamp_ge_P95',
        'max_value_in_window','ratio_max_over_P95','count_points_le_P5','earliest_timestamp_le_P5',
        'min_value_in_window','ratio_min_over_P5'
    ])

# Merge global thresholds
agg = agg.merge(global_q, on=['cmdb_id', 'kpi_name'], how='left')

# Format earliest timestamps as compact strings
if not agg.empty:
    agg['earliest_timestamp_ge_P95'] = agg['earliest_timestamp_ge_P95'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg['earliest_timestamp_le_P5']  = agg['earliest_timestamp_le_P5'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 4A) Table A: top 20 by ratio_max_over_P95 desc, only include pairs with count_points_ge_P95 >= 3
table_A = agg[agg['count_points_ge_P95'] >= 3].copy()
table_A = table_A.sort_values(by='ratio_max_over_P95', ascending=False)
table_A = table_A[['cmdb_id','kpi_name','global_P95','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window','ratio_max_over_P95']].head(20)

# 4B) Table B: top 20 by ratio_min_over_P5 ascending, only include pairs with count_points_le_P5 >= 3
table_B = agg[agg['count_points_le_P5'] >= 3].copy()
table_B = table_B.sort_values(by='ratio_min_over_P5', ascending=True)
table_B = table_B[['cmdb_id','kpi_name','global_P5','count_points_le_P5','earliest_timestamp_le_P5','min_value_in_window','ratio_min_over_P5']].head(20)

# Tidy numeric formatting for compactness
for col in ['global_P95','max_value_in_window','ratio_max_over_P95']:
    if col in table_A.columns and not table_A.empty:
        table_A[col] = table_A[col].round(6)
for col in ['global_P5','min_value_in_window','ratio_min_over_P5']:
    if col in table_B.columns and not table_B.empty:
        table_B[col] = table_B[col].round(6)

# Additional: summary line listing KPI name keyword patterns present (case-insensitive)
keywords = ['cpu','memory','mem','network','net','packet','latency','loss','io','disk','read','write','error','success','ss','start_time','duration','trace','heap','heapmemory','gc']
present = [kw for kw in keywords if df['kpi_name'].str.contains(kw, case=False, na=False).any()]
summary_line = "Matched KPI keyword patterns: " + (", ".join(present) if present else "none")

# Final compact outputs (max 20 rows each)
table_A, table_B, summary_line

```
Out[2]:
```
Summary of results (compact):

- Table A — top pairs by ratio_max_over_P95 (only pairs with >=3 points >= P95):
  - Many top offenders are container CPU metrics and throttling metrics on specific pods:
    - paymentservice-1 · container.node-5.container_cpu_system_seconds — global_P95 = 0.000000; count >=P95 = 3; earliest = 2022-03-21 08:44:00+08; max = 0.01. (Ratio is not meaningful because P95 = 0.)
    - adservice-2 · container.node-5.container_cpu_system_seconds — global_P95 = 0.015; count = 3; earliest = 08:30:00+08; max = 2.97; ratio ≈ 198.
    - adservice-2 · container.node-5.container_cpu_cfs_throttled_periods — global_P95 = 1.5; count = 3; earliest = 08:30:00+08; max = 68.5; ratio ≈ 45.7.
    - adservice-1 · container.node-5.container_cpu_user_seconds — global_P95 = 0.035; count = 3; earliest = 08:30:00+08; max = 0.94; ratio ≈ 26.9.
    - adservice-2 · container.node-5.container_cpu_user_seconds — global_P95 = 0.165; count = 3; earliest = 08:30:00+08; max = 1.785; ratio ≈ 10.8.
  - Other notable top entries include node-2 system.io.avg_q_sz (ratio 8.0), network retransmission on node-4, and several mesh/trace latency-like metrics. Most earliest exceedances occur between 08:30 and 08:46 (UTC+8).
  - Note: any row with global_P95 == 0 produces an invalid/meaningless ratio and should be treated with caution.

- Table B — top pairs by ratio_min_over_P5 (only pairs with >=3 points <= P5):
  - Many metrics hit zero (min_value_in_window = 0) multiple times in the window, yielding ratio_min_over_P5 = 0:
    - shippingservice-0 · container_memory_failures.* (multiple failure-related KPIs) — global_P5 = 2.0; count_le_P5 = 3; earliest = 08:35:00+08; min = 0.
    - Several paymentservice and checkoutservice mesh destination/source metrics have high global_P5 but min = 0 across multiple points (count_le_P5 >=3).
    - Multiple container_cpu_cfs_periods and mesh-related metrics show repeated zero values in the window.
  - Overall pattern: many network/mesh and some memory/failure metrics dropped to zero repeatedly during the incident window.

- Short keyword summary (KPI name patterns present in metric.csv, case-insensitive):
  cpu, memory, mem, network, net, packet, io, disk, read, write, error, ss, start_time, duration, heap, heapmemory, gc

Concise interpretation:
- The incident window (2022-03-21 08:30–09:00 UTC+8) shows pronounced CPU-related anomalies (many pods with multiple samples >= their global P95) and several throttling/IO/network indicators elevated relative to historical P95.
- Simultaneously, many metrics (mesh/destination, some failure counters) hit zero multiple times (min = 0), which may indicate dropped traffic, resets, or counters returning to zero.
- Treat ratios computed against P95 = 0 as unreliable; focus on entries with positive P95 for meaningful ratio comparisons.

The original code execution output of IPython Kernel is also provided below for reference:

(                       cmdb_id                                           kpi_name  global_P95  count_points_ge_P95 earliest_timestamp_ge_P95  max_value_in_window  ratio_max_over_P95
3612          paymentservice-1      container.node-5.container_cpu_system_seconds    0.000000                    3  2022-03-21 08:44:00+0800             0.010000        2.251800e+13
517                adservice-2      container.node-5.container_cpu_system_seconds    0.015000                    3  2022-03-21 08:30:00+0800             2.970000        1.980000e+02
514                adservice-2  container.node-5.container_cpu_cfs_throttled_p...    1.500000                    3  2022-03-21 08:30:00+0800            68.500000        4.566667e+01
433                adservice-1        container.node-5.container_cpu_user_seconds    0.035000                    3  2022-03-21 08:30:00+0800             0.940000        2.685714e+01
519                adservice-2        container.node-5.container_cpu_user_seconds    0.165000                    3  2022-03-21 08:30:00+0800             1.785000        1.081818e+01
3256                    node-2                                 system.io.avg_q_sz    0.020000                    4  2022-03-21 08:46:00+0800             0.160000        8.000000e+00
347                adservice-0        container.node-5.container_cpu_user_seconds    0.155000                    3  2022-03-21 08:30:00+0800             1.185000        7.645161e+00
4421   recommendationservice-1  mesh.source.recommendationservice.jaeger-colle...    0.001050                    3  2022-03-21 08:52:00+0800             0.003050        2.904762e+00
4029   productcatalogservice-1  mesh.destination.recommendationservice2.produc...    0.000525                    4  2022-03-21 08:39:00+0800             0.001050        2.000000e+00
4167  productcatalogservice2-0  container.node-6.container_cpu_cfs_throttled_p...    0.500000                    5  2022-03-21 08:33:00+0800             1.000000        2.000000e+00
1793        checkoutservice2-0  container.node-5.container_cpu_cfs_throttled_p...    0.500000                    5  2022-03-21 08:30:00+0800             1.000000        2.000000e+00
2448           emailservice2-0      container.node-5.container_cpu_system_seconds    0.005000                    6  2022-03-21 08:30:00+0800             0.010000        2.000000e+00
3544          paymentservice-0        container.node-5.container_cpu_user_seconds    0.005000                    5  2022-03-21 08:30:00+0800             0.010000        2.000000e+00
3750         paymentservice2-0        container.node-6.container_cpu_user_seconds    0.005000                    4  2022-03-21 08:31:00+0800             0.010000        2.000000e+00
3680          paymentservice-2        container.node-5.container_cpu_user_seconds    0.005000                    3  2022-03-21 08:35:00+0800             0.010000        2.000000e+00
4706         shippingservice-1      container.node-5.container_cpu_system_seconds    0.010000                    3  2022-03-21 08:38:00+0800             0.020000        2.000000e+00
4501   recommendationservice-2  mesh.source.recommendationservice.jaeger-colle...    0.000525                    3  2022-03-21 08:46:00+0800             0.001050        2.000000e+00
3400                    node-4                        system.net.tcp.retrans_segs    0.030000                    4  2022-03-21 08:32:00+0800             0.050000        1.666667e+00
1031              adservice2-0  mesh.source.adservice2.jaeger-collector.istio_...    0.001575                    3  2022-03-21 08:33:00+0800             0.002625        1.666667e+00
3603          paymentservice-0  mesh.destination.checkoutservice.paymentservic...    0.001050                    3  2022-03-21 08:35:00+0800             0.001575        1.500000e+00,                 cmdb_id                                           kpi_name   global_P5  count_points_le_P5  earliest_timestamp_le_P5  min_value_in_window  ratio_min_over_P5
4652  shippingservice-0  container.node-5.container_memory_failures.con...     2.00000                   3  2022-03-21 08:35:00+0800                  0.0                0.0
4654  shippingservice-0  container.node-5.container_memory_failures.hie...     2.00000                   3  2022-03-21 08:35:00+0800                  0.0                0.0
3668   paymentservice-1  mesh.destination.checkoutservice.paymentservic...   332.50000                   6  2022-03-21 08:30:00+0800                  0.0                0.0
3672   paymentservice-1  mesh.destination.checkoutservice.paymentservic...   675.00000                   6  2022-03-21 08:30:00+0800                  0.0                0.0
3674   paymentservice-2         container.node-5.container_cpu_cfs_periods     1.00000                   9  2022-03-21 08:30:00+0800                  0.0                0.0
3673   paymentservice-1  mesh.destination.checkoutservice.paymentservic...     0.50000                   5  2022-03-21 08:30:00+0800                  0.0                0.0
3671   paymentservice-1  mesh.destination.checkoutservice.paymentservic...     0.50000                   5  2022-03-21 08:30:00+0800                  0.0                0.0
3608   paymentservice-1         container.node-5.container_cpu_cfs_periods     1.00000                   8  2022-03-21 08:32:00+0800                  0.0                0.0
3670   paymentservice-1  mesh.destination.checkoutservice.paymentservic...     0.50000                   5  2022-03-21 08:30:00+0800                  0.0                0.0
1492  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     0.50000                   5  2022-03-21 08:37:00+0800                  0.0                0.0
1494  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     0.50000                   5  2022-03-21 08:37:00+0800                  0.0                0.0
1497  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...   725.00000                   5  2022-03-21 08:37:00+0800                  0.0                0.0
1499  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     0.50000                   5  2022-03-21 08:37:00+0800                  0.0                0.0
1501  checkoutservice-0  mesh.source.checkoutservice.cartservice.istio_...  1650.00000                   5  2022-03-21 08:37:00+0800                  0.0                0.0
431         adservice-1      container.node-5.container_cpu_system_seconds     0.00500                   4  2022-03-21 08:31:00+0800                  0.0                0.0
1487  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...   392.50000                   5  2022-03-21 08:37:00+0800                  0.0                0.0
1490  checkoutservice-0  mesh.destination.frontend.checkoutservice.isti...     0.01825                   3  2022-03-21 08:37:00+0800                  0.0                0.0
1504  checkoutservice-0  mesh.source.checkoutservice.cartservice.istio_...     0.00105                   4  2022-03-21 08:37:00+0800                  0.0                0.0
1506  checkoutservice-0  mesh.source.checkoutservice.cartservice.istio_...     1.00000                   5  2022-03-21 08:37:00+0800                  0.0                0.0
2366     emailservice-1  mesh.source.emailservice.jaeger-collector.isti...   975.00000                   7  2022-03-21 08:38:00+0800                  0.0                0.0, 'Matched KPI keyword patterns: cpu, memory, mem, network, net, packet, io, disk, read, write, error, ss, start_time, duration, heap, heapmemory, gc')```
```

In [ ]:
# In[3]:

import pandas as pd
import pytz
import numpy as np

# Reuse timezone and window if available, else define
try:
    tz
except NameError:
    tz = pytz.timezone('Asia/Shanghai')

start = pd.Timestamp('2022-03-21 08:30:00', tz=tz)
end   = pd.Timestamp('2022-03-21 09:00:00', tz=tz)

# Load trace.csv
tr = pd.read_csv('trace.csv')

# Convert timestamp to timezone-aware datetimes (UTC -> Asia/Shanghai)
tr['dt_utc'] = pd.to_datetime(tr['timestamp'], unit='s', utc=True)
tr['dt'] = tr['dt_utc'].dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) using full series (no pre-filter)
global_p95_trace = tr.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).rename('global_P95').reset_index()

# 2) Filter rows to the incident window (inclusive)
tr_window = tr[(tr['dt'] >= start) & (tr['dt'] <= end)].copy()

# Merge global P95 into the window rows
tr_window = tr_window.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 3) Aggregate per (cmdb_id, trace_name) inside the window
def agg_trace(g):
    cnt = int(len(g))
    gp95 = float(g['global_P95'].iloc[0]) if pd.notna(g['global_P95'].iloc[0]) else np.nan
    mask_ge = g['value'] >= gp95
    cnt_ge = int(mask_ge.sum())
    earliest_ge = g.loc[mask_ge, 'dt'].min() if cnt_ge > 0 else pd.NaT
    max_val = float(g['value'].max()) if cnt > 0 else np.nan
    return pd.Series({
        'count_points_in_window': cnt,
        'count_points_ge_P95': cnt_ge,
        'earliest_timestamp_ge_P95': earliest_ge,
        'max_value_in_window': max_val,
        'global_P95': gp95
    })

if not tr_window.empty:
    agg_trace_df = tr_window.groupby(['cmdb_id', 'trace_name']).apply(agg_trace).reset_index()
else:
    agg_trace_df = pd.DataFrame(columns=['cmdb_id','trace_name','count_points_in_window','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window','global_P95'])

# Format earliest timestamps compactly
if not agg_trace_df.empty:
    agg_trace_df['earliest_timestamp_ge_P95'] = agg_trace_df['earliest_timestamp_ge_P95'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# 4) Top 20 rows sorted by count_points_ge_P95 descending (tie-breaker: max_value_in_window desc)
top_traces = agg_trace_df.sort_values(by=['count_points_ge_P95', 'max_value_in_window'], ascending=[False, False]) \
                         [['cmdb_id','trace_name','global_P95','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window']].head(20)

# Tidy numeric columns for compact display
if not top_traces.empty:
    top_traces['global_P95'] = top_traces['global_P95'].round(6)
    top_traces['max_value_in_window'] = top_traces['max_value_in_window'].round(6)

# Also return list of unique trace_name patterns (top 50 by frequency) to help identify latency/duration/success traces
trace_name_counts = tr['trace_name'].value_counts().reset_index().rename(columns={'index':'trace_name','trace_name':'count'}).head(50)

# Final compact outputs
top_traces, trace_name_counts

```
Out[3]:
```
- Data/Window: analyzed trace.csv for the incident window 2022-03-21 08:30:00–09:00:00 (UTC+8).

- Top results (by count_points_ge_P95):
  - The top 20 (cmdb_id, trace_name) all have count_points_ge_P95 = 30 and earliest exceedance = 2022-03-21 08:30:00+0800.
  - Prominent entries include cartservice (trace.self.duration_mean and duration_p95), cartservice pods and frontend → cartservice traces (duration_mean / duration_p95), and many per-pod error_rate traces (adservice*, cartservice*, adservice2-0, etc.).
  - Max values in the window are small (e.g., durations in the 0.0001–0.0079 sec range; many error_rate max = 0.0).

- Important caveat:
  - Many reported global_P95 values are 0.0. When global_P95 == 0, the condition value >= P95 is trivially true for all non-negative values, so count_points_ge_P95 = full-sample-count (30). Thus these “anomalies” are likely an artifact of a zero threshold rather than meaningful latency spikes. Treat entries with global_P95 > 0 as more informative.

- Trace-name pattern summary (top patterns by frequency — examples present in the data):
  - trace.from_frontend-0.duration_mean, trace.from_frontend-0.duration_p95, trace.from_frontend-0.error_rate, trace.from_frontend-0.row_count
  - Similar families for from_frontend-1/2, from_frontend2-0, to_cartservice-1 (duration_mean/p95), trace.self.duration_mean / duration_p95
  - In short: many duration/mean/p95 traces and many error_rate and row_count traces across services/pods.

- Interpretation:
  - The trace data shows many duration and error_rate traces hitting or exceeding their stored global P95 throughout the entire incident window, but because many P95 thresholds are zero, the counts are not reliable indicators of true latency anomalies. Focus follow-up on traces with positive global_P95 and elevated max values (and on error_rate traces with non-zero values).

The original code execution output of IPython Kernel is also provided below for reference:

(            cmdb_id                            trace_name  global_P95  count_points_ge_P95 earliest_timestamp_ge_P95  max_value_in_window
65    cartservice-0               trace.self.duration_p95         0.0                   30  2022-03-21 08:30:00+0800             0.007950
64    cartservice-0              trace.self.duration_mean         0.0                   30  2022-03-21 08:30:00+0800             0.003026
85    cartservice-1    trace.from_frontend-1.duration_p95         0.0                   30  2022-03-21 08:30:00+0800             0.000650
685      frontend-1   trace.to_cartservice-1.duration_p95         0.0                   30  2022-03-21 08:30:00+0800             0.000650
133  cartservice2-0               trace.self.duration_p95         0.0                   30  2022-03-21 08:30:00+0800             0.000400
84    cartservice-1   trace.from_frontend-1.duration_mean         0.0                   30  2022-03-21 08:30:00+0800             0.000125
684      frontend-1  trace.to_cartservice-1.duration_mean         0.0                   30  2022-03-21 08:30:00+0800             0.000125
2       adservice-0      trace.from_frontend-0.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
6       adservice-0      trace.from_frontend-1.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
10      adservice-0      trace.from_frontend-2.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
14      adservice-1      trace.from_frontend-0.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
18      adservice-1      trace.from_frontend-1.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
22      adservice-1      trace.from_frontend-2.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
26      adservice-2      trace.from_frontend-0.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
30      adservice-2      trace.from_frontend-1.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
34      adservice-2      trace.from_frontend-2.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
38     adservice2-0     trace.from_frontend2-0.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
54    cartservice-0      trace.from_frontend-0.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
58    cartservice-0      trace.from_frontend-1.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000
62    cartservice-0      trace.from_frontend-2.error_rate         0.0                   30  2022-03-21 08:30:00+0800             0.000000,                                        count  count
0        trace.from_frontend-0.duration_mean    587
1         trace.from_frontend-0.duration_p95    587
2           trace.from_frontend-0.error_rate    587
3            trace.from_frontend-0.row_count    587
4            trace.from_frontend-2.row_count    580
..                                       ...    ...
45       trace.from_frontend2-0.duration_p95    204
46         trace.from_frontend2-0.error_rate    204
47          trace.from_frontend2-0.row_count    204
48  trace.to_currencyservice-1.duration_mean    170
49   trace.to_currencyservice-1.duration_p95    170

[50 rows x 2 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd
import pytz
import numpy as np

# Timezone and window (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
start = pd.Timestamp('2022-03-21 08:30:00', tz=tz)
end   = pd.Timestamp('2022-03-21 09:00:00', tz=tz)

# 1) Load log.csv and compute global P95 per (cmdb_id, log_name)
log_df = pd.read_csv('log.csv')
log_df['dt_utc'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)
log_df['dt'] = log_df['dt_utc'].dt.tz_convert(tz)

global_p95_log = log_df.groupby(['cmdb_id','log_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_P95'})
# merge global P95 back
log_df = log_df.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')

# Filter to incident window
log_win = log_df[(log_df['dt'] >= start) & (log_df['dt'] <= end)].copy()

# Aggregate per (cmdb_id, log_name) inside window
def agg_log(g):
    cnt = int(len(g))
    gp95 = float(g['global_P95'].iloc[0]) if pd.notna(g['global_P95'].iloc[0]) else np.nan
    mask = g['value'] >= gp95
    cnt_ge = int(mask.sum())
    earliest_ge = g.loc[mask, 'dt'].min() if cnt_ge>0 else pd.NaT
    max_val = float(g['value'].max()) if cnt>0 else np.nan
    return pd.Series({
        'count_points_in_window': cnt,
        'count_points_ge_P95': cnt_ge,
        'earliest_timestamp_ge_P95': earliest_ge,
        'max_value_in_window': max_val,
        'global_P95': gp95
    })

if not log_win.empty:
    agg_log_df = log_win.groupby(['cmdb_id','log_name']).apply(agg_log).reset_index()
else:
    agg_log_df = pd.DataFrame(columns=['cmdb_id','log_name','count_points_in_window','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window','global_P95'])

# Format earliest timestamps compactly
if not agg_log_df.empty:
    agg_log_df['earliest_timestamp_ge_P95'] = agg_log_df['earliest_timestamp_ge_P95'].dt.strftime('%Y-%m-%d %H:%M:%S%z')

# Top 20 by count_points_ge_P95 desc, tie-breaker max_value_in_window desc
log_top20 = agg_log_df.sort_values(by=['count_points_ge_P95','max_value_in_window'], ascending=[False, False]) \
    [['cmdb_id','log_name','global_P95','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window']].head(20)

# Tidy numeric columns
if not log_top20.empty:
    log_top20['global_P95'] = log_top20['global_P95'].round(6)
    log_top20['max_value_in_window'] = log_top20['max_value_in_window'].round(6)

# 2) Load error_logs.csv and analyze within incident window
err = pd.read_csv('error_logs.csv')
err['dt_utc'] = pd.to_datetime(err['timestamp'], unit='s', utc=True)
err['dt'] = err['dt_utc'].dt.tz_convert(tz)

err_win = err[(err['dt'] >= start) & (err['dt'] <= end)].copy()

total_error_count = int(len(err_win))
unique_err_cmdb = int(err_win['cmdb_id'].nunique())

# Top 10 cmdb_id by error count with earliest error timestamp
if not err_win.empty:
    err_group = err_win.groupby('cmdb_id').agg(
        error_count_in_window = ('message','count'),
        earliest_error_timestamp = ('dt','min')
    ).reset_index()
    err_group['earliest_error_timestamp'] = err_group['earliest_error_timestamp'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    top10_errors = err_group.sort_values(by='error_count_in_window', ascending=False).head(10)
else:
    top10_errors = pd.DataFrame(columns=['cmdb_id','error_count_in_window','earliest_error_timestamp'])

# For top 5 cmdb_id by error_count, collect up to 5 example messages (trimmed)
top5_ids = top10_errors['cmdb_id'].head(5).tolist()
examples = []
for cid in top5_ids:
    msgs = err_win.loc[err_win['cmdb_id']==cid, 'message'].dropna().astype(str)
    # trim whitespace and limit length per example
    uniq_msgs = msgs.drop_duplicates().head(5).apply(lambda s: s.strip().replace('\n',' ').replace('\r',' '))
    joined = ' || '.join(uniq_msgs.tolist())
    examples.append((cid, joined))

examples_df = pd.DataFrame(examples, columns=['cmdb_id','example_messages']).head(5)

# 3) 1-line summary listing any cmdb_id(s) that appear in both metric-level candidates and top error_log list
# Metric-level candidates prefixes: adservice*, paymentservice*, cartservice*, node-*
metric_candidate_prefixes = ['adservice','paymentservice','cartservice','node-']
top_err_ids = top10_errors['cmdb_id'].tolist()
overlap = []
for cid in top_err_ids:
    if any(cid.startswith(p) for p in metric_candidate_prefixes):
        overlap.append(cid)
overlap_summary = "Overlapping cmdb_id(s): " + (", ".join(overlap) if overlap else "none")

# Final compact outputs (as variables)
log_top20, total_error_count, unique_err_cmdb, top10_errors, examples_df, overlap_summary

```
Out[4]:
```
Summary (incident window 2022-03-21 08:30–09:00 UTC+8):

1) log.csv (top anomalies by count_points_ge_P95)
- Many log.error_count series have global_P95 = 0.0, so value >= P95 is trivially true; these show count_points_ge_P95 = 30 (full window) and max_value_in_window = 0.0 for many services (e.g., adservice-0/1/2, cartservice-0/1/2, productcatalogservice-*, recommendationservice-*, shippingservice-*). paymentservice-0 shows 29 points >= P95.
- Note: global_P95 == 0 makes these “anomalies” not meaningful — treat with caution.

2) error_logs.csv (window summary)
- Total error log entries in window: 627
- Unique cmdb_id with errors: 3
- Top 3 cmdb_id by error count (top 10 shown in execution; top 3 are):
  - frontend-0: 235 errors, earliest 2022-03-21 08:30:05+0800
  - frontend-2: 229 errors, earliest 2022-03-21 08:30:18+0800
  - frontend-1: 163 errors, earliest 2022-03-21 08:30:07+0800
- Example messages for the top cmdb_ids (up to 5 examples per component, trimmed):
  - frontend-0: "severity: warning, message: failed to retrieve..." (examples show failed retrieval warnings)
  - frontend-2: "severity: warning, message: failed to retrieve..."
  - frontend-1: "severity: warning, message: failed to retrieve..."

3) Overlap with metric-level candidates
- No overlapping cmdb_id found between the metric-level candidate prefixes (adservice*, paymentservice*, cartservice*, node-*) and the top error_log cmdb_id list (frontends).

Concise interpretation:
- Log metric thresholds (P95) are mostly zero, so log-derived “anomalies” are not reliable here.
- The real error signal is in error_logs: the three frontend pods account for all 627 errors during the window with many “failed to retrieve” warnings; investigate frontend components and their upstream calls.

The original code execution output of IPython Kernel is also provided below for reference:

(                    cmdb_id         log_name  global_P95  count_points_ge_P95 earliest_timestamp_ge_P95  max_value_in_window
0               adservice-0  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
2               adservice-1  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
4               adservice-2  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
6             cartservice-0  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
8             cartservice-1  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
10            cartservice-2  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
16        checkoutservice-2  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
18        currencyservice-0  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
20        currencyservice-1  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
22        currencyservice-2  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
42  productcatalogservice-0  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
44  productcatalogservice-1  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
46  productcatalogservice-2  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
48  recommendationservice-0  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
50  recommendationservice-1  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
52  recommendationservice-2  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
56        shippingservice-0  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
58        shippingservice-1  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
60        shippingservice-2  log.error_count         0.0                   30  2022-03-21 08:30:00+0800                  0.0
36         paymentservice-0  log.error_count         0.0                   29  2022-03-21 08:30:00+0800                  0.0, 627, 3,       cmdb_id  error_count_in_window  earliest_error_timestamp
0  frontend-0                    235  2022-03-21 08:30:05+0800
2  frontend-2                    229  2022-03-21 08:30:18+0800
1  frontend-1                    163  2022-03-21 08:30:07+0800,       cmdb_id                                   example_messages
0  frontend-0  severity: warning, message: failed to retrieve...
1  frontend-2  severity: warning, message: failed to retrieve...
2  frontend-1  severity: warning, message: failed to retrieve..., 'Overlapping cmdb_id(s): none')```
```

In [ ]:
# In[5]:

import pandas as pd
import pytz
import numpy as np

# Reuse existing variables if present, else load trace.csv
try:
    tr
except NameError:
    tr = pd.read_csv('trace.csv')

# Ensure timezone-aware datetimes in UTC then convert to Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in tr.columns:
    tr['dt_utc'] = pd.to_datetime(tr['timestamp'], unit='s', utc=True)
    tr['dt'] = tr['dt_utc'].dt.tz_convert(tz)

# 1) Compute global P95 per (cmdb_id, trace_name) using full series (do NOT filter by window)
try:
    global_p95_trace
except NameError:
    global_p95_trace = tr.groupby(['cmdb_id', 'trace_name'])['value'].quantile(0.95).rename('global_P95').reset_index()

# 2) Filter rows to the incident window (inclusive)
start = pd.Timestamp('2022-03-21 08:30:00', tz=tz)
end   = pd.Timestamp('2022-03-21 09:00:00', tz=tz)
tr_window = tr[(tr['dt'] >= start) & (tr['dt'] <= end)].copy()

# Merge global P95 into window rows
tr_window = tr_window.merge(global_p95_trace, on=['cmdb_id', 'trace_name'], how='left')

# 3) For each pair in the window compute required metrics
def agg_fn(g):
    gp95 = float(g['global_P95'].iloc[0]) if pd.notna(g['global_P95'].iloc[0]) else np.nan
    cnt_in_win = int(len(g))
    mask = g['value'] >= gp95 if not np.isnan(gp95) else pd.Series([False]*len(g), index=g.index)
    cnt_ge = int(mask.sum())
    earliest_ge = g.loc[mask, 'dt'].min() if cnt_ge>0 else pd.NaT
    max_val = float(g['value'].max()) if cnt_in_win>0 else np.nan
    return pd.Series({
        'global_P95': gp95,
        'count_points_in_window': cnt_in_win,
        'count_points_ge_P95': cnt_ge,
        'earliest_timestamp_ge_P95': earliest_ge,
        'max_value_in_window': max_val
    })

if not tr_window.empty:
    agg = tr_window.groupby(['cmdb_id','trace_name']).apply(agg_fn).reset_index()
else:
    agg = pd.DataFrame(columns=['cmdb_id','trace_name','global_P95','count_points_in_window','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window'])

# Filter: global_P95 > 0 and count_points_ge_P95 >= 1
agg_filtered = agg[(agg['global_P95'] > 0) & (agg['count_points_ge_P95'] >= 1)].copy()

# Format earliest timestamp and tidy numeric columns
if not agg_filtered.empty:
    agg_filtered['earliest_timestamp_ge_P95'] = agg_filtered['earliest_timestamp_ge_P95'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
    agg_filtered['global_P95'] = agg_filtered['global_P95'].round(6)
    agg_filtered['max_value_in_window'] = agg_filtered['max_value_in_window'].round(6)

# Sort by earliest_timestamp_ge_P95 ascending and limit to top 20
# Convert back to datetime for proper sorting; NaT rows will be placed at end
if not agg_filtered.empty:
    agg_filtered['__sort_dt'] = pd.to_datetime(agg_filtered['earliest_timestamp_ge_P95'], format='%Y-%m-%d %H:%M:%S%z', utc=True)
    top20 = agg_filtered.sort_values(by='__sort_dt', ascending=True).head(20)
    top20 = top20[['cmdb_id','trace_name','global_P95','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window']].reset_index(drop=True)
    top20 = top20.copy()
    top_count = top20.shape[0]
else:
    top20 = agg_filtered[['cmdb_id','trace_name','global_P95','count_points_ge_P95','earliest_timestamp_ge_P95','max_value_in_window']].head(0)
    top_count = 0

count_line = f"Distinct (cmdb_id, trace_name) pairs with global_P95>0 and >=1 point >=P95 in window: {int(agg_filtered.shape[0])}"

# Final compact outputs
top20, count_line

```
Out[5]:
```
Summary (window 2022-03-21 08:30–09:00 UTC+8):

- Selection: pairs where global_P95 > 0 and at least one point in-window >= global_P95.
- Total distinct (cmdb_id, trace_name) matching: 877.

Top examples (first 20 by earliest exceedance, earliest exceedances at 2022-03-21 08:30:00+0800 unless noted):
- shippingservice2-0 · trace.from_frontend2-0.row_count — global_P95=10.0; count_ge_P95=3; earliest=08:30; max_in_window=10
- checkoutservice-1 · trace.to_emailservice-2.row_count — global_P95=1.0; count_ge_P95=14; earliest=08:30; max=1
- frontend2-0 · trace.to_checkoutservice2-0.row_count — global_P95=2.0; count_ge_P95=5; earliest=08:30; max=2
- checkoutservice-1 · trace.to_paymentservice-0.row_count — global_P95=1.0; count_ge_P95=14; earliest=08:30; max=1
- frontend2-0 · trace.to_checkoutservice2-0.duration_p95 — global_P95≈0.09768; count_ge_P95=2; earliest=08:30; max≈0.15640
- frontend2-0 · trace.to_checkoutservice2-0.duration_mean — global_P95≈0.09566; count_ge_P95=2; earliest=08:30; max≈0.11952
- checkoutservice-1 · trace.to_paymentservice-1.row_count — global_P95=1.0; count_ge_P95=14; earliest=08:30; max=1
- frontend2-0 · trace.to_cartservice2-0.duration_p95 — global_P95=0.001000; count_ge_P95=29; earliest=08:30; max=0.001000
- frontend2-0 · trace.to_adservice2-0.duration_mean — global_P95≈0.000079; count_ge_P95=2; earliest=08:30; max≈0.000083
- checkoutservice-1 · trace.to_productcatalogservice-0.duration_p95 — global_P95≈0.00514; count_ge_P95=2; earliest=08:30; max≈0.005206
- checkoutservice-1 · trace.to_productcatalogservice-0.row_count — global_P95=2.0; count_ge_P95=4; earliest=08:30; max=3
- checkoutservice-1 · trace.to_emailservice-1.row_count — global_P95=1.0; count_ge_P95=15; earliest=08:30; max=1
- checkoutservice-1 · trace.to_productcatalogservice-1.row_count — global_P95=2.0; count_ge_P95=4; earliest=08:30; max=2
- frontend-2 · trace.to_currencyservice-2.row_count — global_P95≈51.55; count_ge_P95=2; earliest=08:30; max=53
- frontend-2 · trace.to_currencyservice-1.row_count — global_P95=52.0; count_ge_P95=3; earliest=08:30; max=53
- frontend-2 · trace.to_currencyservice-1.duration_p95 — global_P95≈0.000128; count_ge_P95=2; earliest=08:30; max≈0.000131
- checkoutservice-1 · trace.to_shippingservice-0.row_count — global_P95=2.0; count_ge_P95=8; earliest=08:30; max=2
- frontend-2 · trace.to_currencyservice-0.row_count — global_P95≈51.55; count_ge_P95=2; earliest=08:30; max=53
- frontend-2 · trace.to_checkoutservice-2.row_count — global_P95=1.0; count_ge_P95=16; earliest=08:30; max=1
- checkoutservice-1 · trace.to_shippingservice-1.row_count — global_P95=2.0; count_ge_P95=7; earliest=08:30; max=2

Interpretation / next steps (concise):
- Many row_count traces exceeded positive historical P95 right at or very near the window start — indicates increased request/traffic to services (notably frontend2, checkoutservice-1, frontend-2, shippingservice2-0).
- Some duration traces (front-end → checkoutservice2, to_currencyservice, etc.) also exceeded their P95s, indicating latency increases for specific call paths.
- Prioritize investigation of frontend2, frontend-2, checkoutservice-1, and shippingservice2-0 call flows (high row_count and some elevated durations).

The original code execution output of IPython Kernel is also provided below for reference:

(               cmdb_id                                     trace_name  global_P95  count_points_ge_P95 earliest_timestamp_ge_P95  max_value_in_window
0   shippingservice2-0               trace.from_frontend2-0.row_count   10.000000                    3  2022-03-21 08:30:00+0800            10.000000
1    checkoutservice-1              trace.to_emailservice-2.row_count    1.000000                   14  2022-03-21 08:30:00+0800             1.000000
2          frontend2-0          trace.to_checkoutservice2-0.row_count    2.000000                    5  2022-03-21 08:30:00+0800             2.000000
3    checkoutservice-1            trace.to_paymentservice-0.row_count    1.000000                   14  2022-03-21 08:30:00+0800             1.000000
4          frontend2-0       trace.to_checkoutservice2-0.duration_p95    0.097681                    2  2022-03-21 08:30:00+0800             0.156399
5          frontend2-0      trace.to_checkoutservice2-0.duration_mean    0.095659                    2  2022-03-21 08:30:00+0800             0.119518
6    checkoutservice-1            trace.to_paymentservice-1.row_count    1.000000                   14  2022-03-21 08:30:00+0800             1.000000
7          frontend2-0           trace.to_cartservice2-0.duration_p95    0.001000                   29  2022-03-21 08:30:00+0800             0.001000
8          frontend2-0            trace.to_adservice2-0.duration_mean    0.000079                    2  2022-03-21 08:30:00+0800             0.000083
9    checkoutservice-1  trace.to_productcatalogservice-0.duration_p95    0.005140                    2  2022-03-21 08:30:00+0800             0.005206
10   checkoutservice-1     trace.to_productcatalogservice-0.row_count    2.000000                    4  2022-03-21 08:30:00+0800             3.000000
11   checkoutservice-1              trace.to_emailservice-1.row_count    1.000000                   15  2022-03-21 08:30:00+0800             1.000000
12   checkoutservice-1     trace.to_productcatalogservice-1.row_count    2.000000                    4  2022-03-21 08:30:00+0800             2.000000
13          frontend-2           trace.to_currencyservice-2.row_count   51.550000                    2  2022-03-21 08:30:00+0800            53.000000
14   checkoutservice-1           trace.to_shippingservice-0.row_count    2.000000                    8  2022-03-21 08:30:00+0800             2.000000
15          frontend-2           trace.to_currencyservice-1.row_count   52.000000                    3  2022-03-21 08:30:00+0800            53.000000
16          frontend-2        trace.to_currencyservice-1.duration_p95    0.000128                    2  2022-03-21 08:30:00+0800             0.000131
17   checkoutservice-1           trace.to_shippingservice-1.row_count    2.000000                    7  2022-03-21 08:30:00+0800             2.000000
18          frontend-2           trace.to_currencyservice-0.row_count   51.550000                    2  2022-03-21 08:30:00+0800            53.000000
19          frontend-2           trace.to_checkoutservice-2.row_count    1.000000                   16  2022-03-21 08:30:00+0800             1.000000, 'Distinct (cmdb_id, trace_name) pairs with global_P95>0 and >=1 point >=P95 in window: 877')```
```

In [ ]:
# In[6]:

import pandas as pd
import pytz
import numpy as np

# Load or reuse metric dataframe
try:
    df
except NameError:
    df = pd.read_csv('metric.csv')

# Ensure timezone-aware datetimes in UTC then convert to Asia/Shanghai (UTC+8)
tz = pytz.timezone('Asia/Shanghai')
if 'dt' not in df.columns:
    df['dt_utc'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)
    df['dt'] = df['dt_utc'].dt.tz_convert(tz)

# Targets
cmdb_ids = ['adservice-2', 'adservice2-0']
kpis = [
    'container.node-5.container_cpu_user_seconds',
    'container.node-5.container_cpu_system_seconds',
    'container.node-5.container_cpu_cfs_throttled_periods'
]

# 1) Compute global P95 for relevant (cmdb_id, kpi_name) using full series (no window filtering)
# Filter full df to matching kpi names first to speed up quantile calc
full_candidates = df[df['kpi_name'].isin(kpis)].copy()
global_p95 = full_candidates.groupby(['cmdb_id','kpi_name'])['value'].quantile(0.95).reset_index().rename(columns={'value':'global_P95'})

# Ensure we have rows for both cmdb_ids even if missing (left join later)
# 2) Filter to incident window (inclusive)
start = pd.Timestamp('2022-03-21 08:30:00', tz=tz)
end   = pd.Timestamp('2022-03-21 09:00:00', tz=tz)
window_df = df[(df['dt'] >= start) & (df['dt'] <= end) & (df['cmdb_id'].isin(cmdb_ids)) & (df['kpi_name'].isin(kpis))].copy()

# Attach global_P95 to window rows by (cmdb_id,kpi_name)
window_df = window_df.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Prepare result rows for each (cmdb_id, kpi)
rows = []
for cid in cmdb_ids:
    for k in kpis:
        gp95_row = global_p95[(global_p95['cmdb_id']==cid) & (global_p95['kpi_name']==k)]
        gp95 = float(gp95_row['global_P95'].iloc[0]) if not gp95_row.empty else np.nan
        subset = window_df[(window_df['cmdb_id']==cid) & (window_df['kpi_name']==k)].copy().sort_values('dt')
        # Identify mask where value >= global_P95 (handle NaN)
        if np.isnan(gp95):
            mask = pd.Series([False]*len(subset), index=subset.index)
        else:
            mask = subset['value'] >= gp95
        subset = subset.assign(anom=mask.astype(int))
        # find consecutive segments where anom==1 and timestamps are consecutive minutes
        if subset.empty or subset['anom'].sum() == 0:
            earliest = pd.NaT
            seg_len = 0
            max_in_seg = np.nan
            percent_over = np.nan
        else:
            # get rows where anom==1
            anom_rows = subset[subset['anom']==1].copy()
            # compute time delta in seconds from previous anom row (within anom_rows)
            anom_rows['ts'] = anom_rows['dt'].astype('int64')//1_000_000_000  # seconds since epoch
            # identify breaks where difference != 60 (non-consecutive minute)
            anom_rows['delta'] = anom_rows['ts'].diff().fillna(0)
            # start new group when delta != 60
            anom_rows['grp'] = (anom_rows['delta'] != 60).cumsum()
            # find groups, take first group's start, length, and max value in that group
            first_grp = anom_rows['grp'].min()
            grp_rows = anom_rows[anom_rows['grp']==first_grp]
            earliest = grp_rows['dt'].min()
            seg_len = int(len(grp_rows))
            max_in_seg = float(grp_rows['value'].max())
            if gp95 == 0 or np.isnan(gp95):
                percent_over = np.nan
            else:
                percent_over = (max_in_seg - gp95)/gp95*100.0
        # append row
        rows.append({
            'cmdb_id': cid,
            'kpi_name': k,
            'global_P95': gp95,
            'earliest_anomaly_start_timestamp_in_window': earliest,
            'anomaly_segment_length_in_minutes': seg_len,
            'max_value_in_that_segment': max_in_seg if not np.isnan(max_in_seg) else np.nan,
            'percent_over_P95': percent_over
        })

result = pd.DataFrame(rows)

# Format timestamps and numeric columns compactly
result['earliest_anomaly_start_timestamp_in_window'] = result['earliest_anomaly_start_timestamp_in_window'].dt.strftime('%Y-%m-%d %H:%M:%S%z')
result['global_P95'] = result['global_P95'].round(6)
result['max_value_in_that_segment'] = result['max_value_in_that_segment'].round(6)
result['percent_over_P95'] = result['percent_over_P95'].round(2)

# Keep only the three KPI rows requested (if duplicates, keep all present combinations). Limit output compactly.
result, result.shape[0]

```
Out[6]:
```
Summary (adservice-2 and adservice2-0, window 2022-03-21 08:30–09:00 UTC+8):

- adservice-2:
  - container.node-5.container_cpu_user_seconds — global_P95 = 0.165. First consecutive anomaly segment starts 2022-03-21 08:30:00+08, length = 1 minute, max in segment = 1.785, percent_over_P95 ≈ 981.82%.
  - container.node-5.container_cpu_system_seconds — global_P95 = 0.015. First segment starts 2022-03-21 08:30:00+08, length = 1 minute, max = 2.97, percent_over_P95 ≈ 19,700%.
  - container.node-5.container_cpu_cfs_throttled_periods — global_P95 = 1.5. First segment starts 2022-03-21 08:30:00+08, length = 1 minute, max = 68.5, percent_over_P95 ≈ 4,466.67%.

- adservice2-0:
  - No global_P95 available for the three requested KPIs in the full series (NaN) and therefore no detected anomaly segments in the incident window.

Interpretation (concise):
- adservice-2 shows single-minute spikes starting exactly at the window start for CPU user, CPU system, and CPU throttling (very large multiples over historical P95). These indicate a brief but strong CPU usage/throttling event on adservice-2 at 08:30. Percent-over-P95 values are extremely large because P95 baselines are very small; treat magnitudes accordingly. adservice2-0 had no matching KPI data here.

The original code execution output of IPython Kernel is also provided below for reference:

(        cmdb_id                                           kpi_name  global_P95 earliest_anomaly_start_timestamp_in_window  anomaly_segment_length_in_minutes  max_value_in_that_segment  percent_over_P95
0   adservice-2        container.node-5.container_cpu_user_seconds       0.165                   2022-03-21 08:30:00+0800                                  1                      1.785            981.82
1   adservice-2      container.node-5.container_cpu_system_seconds       0.015                   2022-03-21 08:30:00+0800                                  1                      2.970          19700.00
2   adservice-2  container.node-5.container_cpu_cfs_throttled_p...       1.500                   2022-03-21 08:30:00+0800                                  1                     68.500           4466.67
3  adservice2-0        container.node-5.container_cpu_user_seconds         NaN                                        NaN                                  0                        NaN               NaN
4  adservice2-0      container.node-5.container_cpu_system_seconds         NaN                                        NaN                                  0                        NaN               NaN
5  adservice2-0  container.node-5.container_cpu_cfs_throttled_p...         NaN                                        NaN                                  0                        NaN               NaN, 6)```
```